# Latent Mesh Diffusion Computer — Phase 1 Training

**Before running:** Get a HF token at `huggingface.co/settings/tokens` (Write scope).

This notebook:
1. Clones the repo
2. Installs dependencies
3. Converts HF datasets → binary shards
4. Trains Phase 1 (250M parameters)
5. Pushes checkpoints to your HF Hub model repo

In [ ]:
# @title 1. Install dependencies
import os, sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch>=2.5', 'transformers', 'datasets', 'numpy',
    'accelerate', 'huggingface-hub', 'tensorboard', 'bitsandbytes'], check=True)
print('Deps installed')

In [ ]:
# @title 2. Clone repo
import os, sys, subprocess
WORK = '/content'
os.chdir(WORK)

REPO = 'https://github.com/mohamed-raad/latent-mesh-diffusion'
if not os.path.isdir('latent-mesh-diffusion'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO], check=True)
os.chdir('latent-mesh-diffusion')
sys.path.insert(0, 'NoProp/src')
print('Repo cloned')

In [ ]:
# @title 3. Login to HuggingFace Hub
from huggingface_hub import login, HfApi
import getpass

HF_TOKEN = getpass.getpass('Enter your HF token:')
HF_REPO_ID = 'mohamed99raad/Latent-Mesh-Model'  # @param {type:"string"}

login(token=HF_TOKEN, add_to_git_credential=True)

api = HfApi()
api.create_repo(HF_REPO_ID, repo_type='model', private=True, exist_ok=True)
print(f'Logged in as {api.whoami()["name"][0]}')
print(f'Model repo: {HF_REPO_ID}')

In [ ]:
# @title 4. Data config (skip — uses direct HF streaming)
from train_pipeline import PHASE_CONFIGS
cfg = PHASE_CONFIGS['phase1_250m']
print('Using direct HF streaming (no conversion needed)')
print('Datasets:', [d['hf_path'] for d in cfg.datasets])

In [ ]:
# @title 5. Train Phase 1 (auto-resume, auto-push to HF Hub)
import time, os
from train_pipeline import TrainingOrchestrator
from huggingface_hub import upload_folder, snapshot_download

LOG_DIR = '/content/logs'
CKPT_DIR = '/content/checkpoints'

# Resume from HF Hub if checkpoints exist there
try:
    snapshot_download(repo_id=HF_REPO_ID, local_dir=CKPT_DIR, token=HF_TOKEN,
                      ignore_patterns=['logs/*'])
    print('Resumed from HF Hub - training continues')
except Exception:
    print('No existing checkpoint - starting fresh')
    os.makedirs(CKPT_DIR, exist_ok=True)

orchestrator = TrainingOrchestrator(
    log_dir=LOG_DIR,
    checkpoint_base=CKPT_DIR,
    use_packing=False,
)

print('Starting Phase 1 (250M) - 50,000 steps')
t0 = time.time()
orchestrator.run_phase(cfg)

# Push final checkpoint to HF Hub
for item in os.listdir(CKPT_DIR):
    item_path = os.path.join(CKPT_DIR, item)
    if os.path.isdir(item_path):
        print(f'Uploading {item}...')
        upload_folder(folder_path=item_path, repo_id=HF_REPO_ID,
                      repo_type='model', path_in_repo=item, token=HF_TOKEN)
upload_folder(folder_path=LOG_DIR, repo_id=HF_REPO_ID,
              repo_type='model', path_in_repo='logs', token=HF_TOKEN)
print(f'All at https://huggingface.co/{HF_REPO_ID}')

elapsed = time.time() - t0
print(f'Train done in {elapsed/3600:.1f}h')

In [ ]:
# @title 7. Check status — test loading the model
from pathlib import Path
print('=== Training Summary ===')
if os.path.isdir(CKPT_DIR):
    phases = [d for d in os.listdir(CKPT_DIR) if d.startswith('phase_')]
    if phases:
        latest = max(phases)
        print(f'✓ Last checkpoint: {latest}')
        print(f'  Finalize at: https://huggingface.co/{HF_REPO_ID}')
    else:
        print('  No checkpoints yet')
else:
    print('  No checkpoints yet')
print(f'\nTo resume if disconnected: re-run Cell 5 (it downloads from HF Hub)')